# G4.P1 — Street Barricade Strategist reconstruction
Run in a **private Kaggle notebook** with **GPU T4 x2 or P100** and **Internet on**. The source image is committed to the game branch. The official SF3D checkpoint is gated: accept its Hugging Face terms yourself, then add a Kaggle secret named `HF_TOKEN` containing a read token. TripoSR runs without this secret. Checkpoints and virtual environments stay in `/kaggle/working/g4p1_results`, outside the game checkout.

The two official `run.py` scripts use the same `front_clean.png`. Source commits are pinned in `kaggle_pipeline.py`; each model uses a separate virtual environment. Read `manifest.json` and the logs before treating either output as complete.


In [ ]:
import os, subprocess, sys, shutil
if not shutil.which('xvfb-run'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'xvfb', 'libgl1-mesa-dri'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU accelerator in Kaggle Settings'
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('SF3D Hugging Face token available.')
except Exception:
    print('No HF_TOKEN secret available; SF3D will be recorded as gated, and TripoSR will still run.')


In [ ]:
from pathlib import Path
workspace = Path('/kaggle/working/CA-VA-PETER-')
if not workspace.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'g4p1-male-mc-reconstruction-pipeline', 'https://github.com/sushan5140/CA-VA-PETER-', str(workspace)], check=True)
subprocess.run([sys.executable, str(workspace / 'tools/g4p1/kaggle_pipeline.py')], cwd=workspace, check=True)


In [ ]:
# Blender is needed for the four-angle renders and later selected-model cleanup.
# Run this cell to render and export candidates after the pipeline.
results = Path('/kaggle/working/g4p1_results')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'blender'], check=True)
for model in ('sf3d', 'triposr'):
    mesh = Path('/kaggle/working/g4p1_results') / model / '0' / 'mesh.glb'
    if mesh.is_file():
        subprocess.run(['blender', '-b', '-t', '2', '--python', str(workspace / 'tools/g4p1/blender_review.py'), '--', '--input', str(mesh), '--render-dir', str(mesh.parent.parent / 'renders'), '--output', str(mesh.parent.parent / f'{model}_candidate.glb')], check=True)
        subprocess.run([str(results / 'venvs' / 'triposr' / 'bin' / 'python'), str(workspace / 'tools/g4p1/validate_glb.py'), str(mesh.parent.parent / f'{model}_candidate.glb'), '--output', str(mesh.parent.parent / 'cleaned_validation.json')], check=True)
comparison = workspace / 'tools/g4p1/make_comparison.py'
subprocess.run([sys.executable, str(comparison), '/kaggle/working/g4p1_results', '--output', '/kaggle/working/g4p1_results/visual_comparison.png'], check=True)


In [ ]:
import json
from IPython.display import display, Image as DisplayImage
results = Path('/kaggle/working/g4p1_results')
print(json.dumps(json.loads((results / 'manifest.json').read_text()), indent=2))
if (results / 'visual_comparison.png').is_file():
    display(DisplayImage(filename=str(results / 'visual_comparison.png')))
print('Review the four views and score both models against the locked G3 sheet before selecting one.')


In [ ]:
# Download only useful evidence; exclude multi-GB model checkpoints and virtual environments.
from zipfile import ZipFile, ZIP_DEFLATED
archive = Path('/kaggle/working/g4p1_deliverables.zip')
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    for file in results.rglob('*'):
        if file.is_file() and not any(part in {'checkpoints', 'venvs', 'upstream'} for part in file.relative_to(results).parts):
            z.write(file, file.relative_to(results))
print('Download from Kaggle Output:', archive)
